# General Guide to Data Cleaning

Data cleaning is the process of detecting and correcting corrupt or inaccurate records from a record set, table, or database and refers to identifying incomplete, incorrect, inaccurate or irrelevant parts of the data and then replacing, modifying, or deleting the dirty or coarse data.

Here are the common steps in data cleaning:
1.  **Load and Inspect Data:** Load your data and get a first overview.
2.  **Handle Missing Values:** Decide on a strategy for dealing with empty cells.
3.  **Remove Duplicates:** Find and remove duplicate rows.
4.  **Validate Data:** Check if the data now meets your requirements.
5.  **Save Data:** Save the cleaned data to a new file.

Let's go through these steps with code examples.

In [7]:
import pandas as pd
import numpy as np

# Load the dataset
df = pd.read_csv('version2/crawl_results.csv')
# df = pd.read_csv('version1_(Abobakr)/metro_products.csv')

print("Original DataFrame:")
df.head()

Original DataFrame:


,product_id,name,price,original_price,discount,brand,category,availability,description,Unnamed: 9,Unnamed: 10,Unnamed: 11,Unnamed: 12,Unnamed: 13,Unnamed: 14,Unnamed: 15,Unnamed: 16,Unnamed: 17,Unnamed: 18
0,25804,Juhayna Full Cream Milk - 1Kg,44.99 LE,52.75 LE,14.7%,Juhayna,Juhayna,in stock,Juhayna Full Cream Milk - 1Kg,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,11760,Crystal Glass Cleaner Spray700+Refil S.O,66.95 LE,79.95 LE,16.3%,Crystal,Crystal,in stock,Crystal Glass Cleaner Spray700+Refil S.O,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,20039,Camay Soap Bar - 165g*4Pc,94.95 LE,104.95 LE,9.5%,Camay,Camay,in stock,Camay Soap Bar - 165g*4Pc,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,20090,Lux Soap Bar Creamy Perfection - 165*4Pc,99.95 LE,114.95 LE,13%,Lux,Lux,in stock,Lux Soap Bar Creamy Perfection - 165*4Pc,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,43172,Lifebuoy Soap Bar 75g*4Pc,54.95 LE,62.95 LE,12.7%,LIFEBUOY,LIFEBUOY,in stock,Lifebuoy Soap Bar 75g*4Pc,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### 1. Load and Inspect Data

First, you need to get a sense of your data's structure and identify potential issues.


In [8]:
# Get a summary of the DataFrame
print("DataFrame Info:")
df.info()

# Get descriptive statistics
print("\nDescriptive Statistics:")
print(df.describe())

# Check for missing values
print("\nMissing Values:")
print(df.isnull().sum())

DataFrame Info:
<class 'pandas.DataFrame'>
RangeIndex: 286 entries, 0 to 285
Data columns (total 19 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   product_id      286 non-null    int64  
 1   name            286 non-null    str    
 2   price           286 non-null    str    
 3   original_price  216 non-null    str    
 4   discount        216 non-null    str    
 5   brand           286 non-null    str    
 6   category        286 non-null    str    
 7   availability    286 non-null    str    
 8   description     286 non-null    str    
 9   Unnamed: 9      0 non-null      float64
 10  Unnamed: 10     0 non-null      float64
 11  Unnamed: 11     21 non-null     str    
 12  Unnamed: 12     9 non-null      str    
 13  Unnamed: 13     8 non-null      str    
 14  Unnamed: 14     8 non-null      str    
 15  Unnamed: 15     8 non-null      str    
 16  Unnamed: 16     0 non-null      float64
 17  Unnamed: 17     8 non-null    

### 2. Handle Missing Values

You can either remove rows with missing values or fill them with a specific value (like the mean, median, or mode).


In [9]:
# Remove unnamed columns
unnamed_cols = [col for col in df.columns if 'Unnamed' in col]
df.drop(columns=unnamed_cols, inplace=True)

# Clean and convert 'price' and 'original_price' to numeric, coercing errors
for col in ['price', 'original_price']:
    if col in df.columns:
        df[col] = df[col].astype(str).str.replace(' LE', '').str.strip()
        df[col] = pd.to_numeric(df[col], errors='coerce')

# Clean and convert 'discount' to numeric, coercing errors
if 'discount' in df.columns:
    df['discount'] = df['discount'].astype(str).str.replace('%', '').str.strip()
    df['discount'] = pd.to_numeric(df['discount'], errors='coerce')

# Drop rows where 'price' is missing
df.dropna(subset=['price'], inplace=True)

# Fill missing 'discount' with 0
if 'discount' in df.columns:
    df['discount'] = df['discount'].fillna(0)

# Fill missing 'original_price'
if 'original_price' in df.columns:
    # Where discount is 0, original_price should be the same as price
    df.loc[df['discount'] == 0, 'original_price'] = df['price']
    
    # For other missing original_price, calculate it if discount is available
    calculated_original_price = df['price'] / (1 - df['discount'] / 100)
    df['original_price'] = df['original_price'].fillna(calculated_original_price)


print("DataFrame after handling missing values and unnamed columns:")
print(df.head())
print("\nMissing values after cleaning:")
print(df.isnull().sum())
# df.head()

DataFrame after handling missing values and unnamed columns:
   product_id                                      name  price  \
0       25804             Juhayna Full Cream Milk - 1Kg  44.99   
1       11760  Crystal Glass Cleaner Spray700+Refil S.O  66.95   
2       20039                 Camay Soap Bar - 165g*4Pc  94.95   
3       20090  Lux Soap Bar Creamy Perfection - 165*4Pc  99.95   
4       43172                 Lifebuoy Soap Bar 75g*4Pc  54.95   

   original_price  discount     brand  category availability  \
0           52.75      14.7   Juhayna   Juhayna     in stock   
1           79.95      16.3   Crystal   Crystal     in stock   
2          104.95       9.5     Camay     Camay     in stock   
3          114.95      13.0       Lux       Lux     in stock   
4           62.95      12.7  LIFEBUOY  LIFEBUOY     in stock   

                                description  
0             Juhayna Full Cream Milk - 1Kg  
1  Crystal Glass Cleaner Spray700+Refil S.O  
2                 C

### 3. Remove Duplicates

Our sample data may have duplicate rows. It's important to remove them to avoid incorrect analysis.

In [10]:
# Remove duplicate rows
df.drop_duplicates(inplace=True)

print("DataFrame after removing duplicates:")
print(df)

DataFrame after removing duplicates:
     product_id                                      name  price  \
0         25804             Juhayna Full Cream Milk - 1Kg  44.99   
1         11760  Crystal Glass Cleaner Spray700+Refil S.O  66.95   
2         20039                 Camay Soap Bar - 165g*4Pc  94.95   
3         20090  Lux Soap Bar Creamy Perfection - 165*4Pc  99.95   
4         43172                 Lifebuoy Soap Bar 75g*4Pc  54.95   
..          ...                                       ...    ...   
279         689                            Parsley - 100g   9.99   
280         691                    Fresh Coriander - 100g   9.99   
281        7818                  Metro Cumin Shaker - 75g  86.99   
282       26639                   Metro Garlic Paste 200g  43.99   
283       24631      Givrex Frozen Chopped Spinach - 400g  24.99   

     original_price  discount          brand       category availability  \
0             52.75      14.7        Juhayna        Juhayna     in sto

### 4. Validate Data

After cleaning, you should validate the data to ensure it's ready for analysis. This can be as simple as running `info()` and `describe()` again, or more complex checks based on your domain knowledge.

In [11]:
print("Final Cleaned DataFrame Info:")
df.info()

print("\nFinal Cleaned DataFrame Description:")
print(df.describe())

print("\nFinal DataFrame:")
print(df)

Final Cleaned DataFrame Info:
<class 'pandas.DataFrame'>
Index: 278 entries, 0 to 283
Data columns (total 9 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   product_id      278 non-null    int64  
 1   name            278 non-null    str    
 2   price           278 non-null    float64
 3   original_price  278 non-null    float64
 4   discount        278 non-null    float64
 5   brand           278 non-null    str    
 6   category        278 non-null    str    
 7   availability    278 non-null    str    
 8   description     278 non-null    str    
dtypes: float64(3), int64(1), str(5)
memory usage: 29.8 KB

Final Cleaned DataFrame Description:
         product_id        price  original_price    discount
count    278.000000   278.000000      278.000000  278.000000
mean   14119.341727   127.699424      150.324676   14.134892
std    10964.977469   152.390031      166.886019   11.403573
min       23.000000     6.500000        7.9500

### 5. Save the Cleaned Data

Now that the data is cleaned, let's save it to a new CSV file.

In [12]:
# Save the cleaned DataFrame to a new CSV file
df.to_csv('handled_data.csv', index=False)

print("Cleaned data saved to 'handled_data.csv'")

Cleaned data saved to 'handled_data.csv'


### 6. Converting to JSON

Now, let's convert the cleaned data to a JSON file with a readable format.

In [13]:
import json

# Convert DataFrame to a list of dictionaries
data_dict = df.to_dict(orient='records')

# Write to a JSON file with indentation
with open('handled_data.json', 'w') as f:
    json.dump(data_dict, f, indent=4)

print("Cleaned data saved to 'handled_data.json' with readable format.")

Cleaned data saved to 'handled_data.json' with readable format.
